# 01 Feature Selection Pipeline Walkthrough

End-to-end documentation notebook for the Feature Selection POC Framework. It demonstrates setup, data loading, metadata, config, quality filters, scorebook, graph backend sync, model pruning, tail analysis, domain rescue, and artifact export.

## 1. Overview

This pipeline is designed for regression feature selection with auditable intermediate artifacts. It is suitable for ordinary and skewed regression and for tail-sensitive tasks. Extremely sparse spike targets should reuse the quality/graph stages but branch to anomaly or rare-event modeling.

Artifacts: selected/removed features, scorebook, metadata, family table, graph nodes/edges, selection history, model evaluation, sync report, and summary markdown.

## 2. Environment Setup

In [ ]:
from pathlib import Path
import logging
import numpy as np
import pandas as pd

from feature_selection_poc.config import FeatureSelectionConfig, ModelConfig, PruningConfig, QualityFilterConfig, ScoringConfig, GraphBackendConfig, OutputConfig
from feature_selection_poc.data import make_regression_poc_dataset
from feature_selection_poc.pipeline import FeatureSelectionPipeline
from feature_selection_poc.kernel.strategies import FeatureQualityFilter, ScorebookBuilder, make_tail_sample_weight
from feature_selection_poc.graph import FeatureGraphBuilder
from feature_selection_poc.graph import cypher_templates

SEED = 42
np.random.seed(SEED)
logging.basicConfig(level=logging.INFO)
OUTPUT_DIR = Path("../outputs/notebook_feature_selection_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load Example Dataset

The framework can use public datasets, but this notebook uses a deterministic synthetic fallback so it runs offline.

In [ ]:
X, y, feature_metadata = make_regression_poc_dataset(n_samples=300, n_features=45, target_type="skewed", random_state=SEED)
print("rows, features:", X.shape)
display(y.describe())
display(X.isna().mean().sort_values(ascending=False).head())
display(X.head())

## 4. Define Feature Metadata

Public data often lacks station/process metadata, so synthetic metadata demonstrates the schema used by graph exports.

In [ ]:
display(feature_metadata.head())
print(feature_metadata.columns.tolist())

## 5. Configure Pipeline

Tune thresholds, model type, pruning count, and graph backend here.

In [ ]:
config = FeatureSelectionConfig(
    quality=QualityFilterConfig(missing_rate_threshold=0.5, near_constant_threshold=0.98, max_features_after_quality=60),
    scoring=ScoringConfig(tail_quantile=0.9),
    pruning=PruningConfig(max_rounds=3, keep_top_k=12, keep_fraction=0.6, min_features=6),
    model=ModelConfig(name="random_forest", params={"n_estimators": 60, "random_state": SEED}, cv_folds=3),
    graph=GraphBackendConfig(backend="local", output_dir=str(OUTPUT_DIR / "graph")),
    output=OutputConfig(output_dir=str(OUTPUT_DIR), run_id="notebook_demo"),
)
config.to_dict()

## 6. Run Feature Quality Filter

Constant, missing, variance, unique-ratio, and target-correlation checks produce removal reasons.

In [ ]:
quality_filter = FeatureQualityFilter(config.quality)
X_quality, quality_metrics = quality_filter.filter(X, y)
print("before:", X.shape[1], "after quality:", X_quality.shape[1])
display(quality_metrics["removal_reason"].value_counts(dropna=False))
display(quality_metrics.head())

## 7. Build Feature Scorebook

The scorebook explains quality, univariate relevance, tail relevance, rescue potential, and final ranking.

In [ ]:
scorebook = ScorebookBuilder(config.scoring).build(X, y, quality_metrics)
display(scorebook.head(10))
display(scorebook[["feature_name", "status", "removal_reason", "final_feature_score", "tail_importance_score"]].tail(10))

## 8. Build Feature Family / Feature Graph

Correlation edges define families; metadata enriches graph nodes for visualization and knowledge graph sync.

In [ ]:
nodes, edges, family = FeatureGraphBuilder(corr_threshold=0.85).build(X, scorebook, feature_metadata)
print("nodes:", nodes.shape, "edges:", edges.shape, "families:", family["family_id"].nunique())
display(family.head())
display(edges.head())

## 9. Sync Feature Graph to Knowledge Graph Backend

Local backend is the default fallback. Neo4j mapping is represented through repository skeleton and Cypher templates.

In [ ]:
pipeline = FeatureSelectionPipeline(config)
print(cypher_templates.FEATURE_NEIGHBORS.strip()[:180], "...")

## 10. Run Model-based Feature Selection

In [ ]:
result = pipeline.fit_select(X, y, feature_metadata=feature_metadata)
print("selected:", result.selected_features)
display(pd.DataFrame([h.to_dict() for h in result.selection_history]))
display(result.scorebook.head(12))

## 11. Tail / Outlier-aware Analysis

In [ ]:
weights = make_tail_sample_weight(y, tail_quantile=config.scoring.tail_quantile, tail_weight=5.0)
print("tail weighted samples:", int((weights > 1).sum()))
display(result.scorebook.nlargest(10, "tail_importance_score")[["feature_name", "tail_importance_score", "status", "removal_reason"]])
print("latest tail metrics:", result.model_evaluation)

## 12. Domain Rescue Experiment

Pick removed high-tail-score features or graph-neighbor features and compare metrics after adding them back.

In [ ]:
rescue_candidates = result.scorebook[result.scorebook["status"].eq("removed")].nlargest(3, "domain_rescue_score")["feature_name"].tolist()
print("rescue candidates:", rescue_candidates)
rescued_result = pipeline.rescue_features(result, rescue_candidates)
rescue_metrics = pipeline.rerun_with_rescued_features(X, y, rescue_candidates) if rescue_candidates else {}
print(rescue_metrics)

## 13. Export Artifacts

In [ ]:
pipeline.save_artifacts(rescued_result, output_dir=OUTPUT_DIR)
repo = pipeline.sync_feature_graph(rescued_result.feature_graph, backend="local")
print("artifact dir:", OUTPUT_DIR.resolve())
print("neighbor query:")
display(repo.query_feature_neighbors(rescued_result.selected_features[0], max_depth=2).head())

## 14. Final Summary

Use the selected features, scorebook, family table, and graph query outputs to review automatic decisions with domain experts. To use your own dataset, replace `X`, `y`, and `feature_metadata`, then keep the config/pipeline calls unchanged.

In [ ]:
print("Final selected features:", rescued_result.selected_features)
print("Model evaluation:", rescued_result.model_evaluation)
display(rescued_result.feature_family.head())